In [0]:
# import pandas as pd
# from pyspark.sql import functions as F

# schema = 'finance'
# table_name = 'dim_taxonomy'

# dbutils.widgets.text("year", "", "GAAP Version Year (leave blank for all versions)")
# gaap_year_to_process = dbutils.widgets.get("year")

# dbutils.widgets.text("target_catalog", "", "Target Catalog")
# target_catalog = dbutils.widgets.get("target_catalog")

# # -----------------------------------------------------------------
# # 1. Load staging data — optionally scoped to a single GAAP version
# # -----------------------------------------------------------------
# staging = spark.table("operations.finance_staging.dim_taxonomy_staging")

# if gaap_year_to_process:
#     gaap_version = f"us-gaap/{gaap_year_to_process}"
#     staging = staging.filter(F.col("gaap_version") == gaap_version)

# df = staging.toPandas()

# # -----------------------------------------------------------------
# # 2. Load fact leaf nodes — version-aware so each (leaf, version)
# #    pair is treated independently
# # -----------------------------------------------------------------
# fact = (
#     spark.table("operations.finance_staging.fact_staging_financial_statement")
#     .select("terse_label", "gaap_version")
#     .distinct()
#     .toPandas()
# )

# # Set of (terse_label, gaap_version) pairs that actually exist in fact
# fact_pairs = set(zip(fact["terse_label"], fact["gaap_version"]))

# # -----------------------------------------------------------------
# # 3. Build parent map — keyed on (child_label, gaap_version, linkrole)
# # -----------------------------------------------------------------
# parent_map = {
#     (row["child_label"], row["gaap_version"], row["linkrole"]): row["parent_label"]
#     for _, row in df.iterrows()
# }

# # -----------------------------------------------------------------
# # 4. Map each child label → all (gaap_version, linkrole) combos it
# #    appears in across the full staging table
# # -----------------------------------------------------------------
# child_version_linkrole_map = (
#     df.groupby("child_label")
#     .apply(lambda x: list(zip(x["gaap_version"], x["linkrole"])))
#     .to_dict()
# )

# # -----------------------------------------------------------------
# # 5. Path builder — unchanged; walks parent_map up to max_depth
# # -----------------------------------------------------------------
# def build_path(child, gaap_version, linkrole, parent_map, max_depth=50):
#     path = []
#     current = child
#     visited = set()
#     for _ in range(max_depth):
#         key = (current, gaap_version, linkrole)
#         if current is None or key in visited:
#             break
#         path.append(current)
#         visited.add(key)
#         current = parent_map.get(key)
#     return path[::-1]

# # -----------------------------------------------------------------
# # 6. Build paths — only for (leaf, version) pairs present in fact,
# #    spanning all linkroles that leaf appears in for that version
# # -----------------------------------------------------------------
# paths = []
# for leaf, version in fact_pairs:
#     version_linkrole_pairs = [
#         (v, lr) for v, lr in child_version_linkrole_map.get(leaf, [])
#         if v == version
#     ]
#     for _, linkrole in version_linkrole_pairs:
#         path = build_path(leaf, version, linkrole, parent_map)
#         paths.append({
#             "leaf_node": leaf,
#             "gaap_version": version,
#             "linkrole": linkrole,
#             "path": path
#         })

# paths_df = pd.DataFrame(paths)
# max_depth = paths_df["path"].apply(len).max()

# # -----------------------------------------------------------------
# # 7. Expand path list into level columns
# #    NOTE: label_map (identity mapping) removed — it was a no-op
# # -----------------------------------------------------------------
# for i in range(max_depth):
#     paths_df[f"level_{i}"] = paths_df["path"].apply(
#         lambda x, i=i: x[i] if i < len(x) else None
#     )

# spark.createDataFrame(paths_df).createOrReplaceTempView('df')

# -----------------------------------------------------------------
# 8. Final select — surrogate keys on leaf, version, and linkrole
# -----------------------------------------------------------------
# final_df = spark.sql(f"""
# select
#      bigint(substr(xxhash64(concat_ws('|', leaf_node)), 1, 18))        AS terse_label_bigint_key
#     ,sha2(concat_ws('|', leaf_node), 256)                               AS terse_label_key_hash
#     ,bigint(substr(xxhash64(concat_ws('|', gaap_version)), 1, 18))      AS gaap_version_bigint_key
#     ,sha2(concat_ws('|', gaap_version), 256)                            AS gaap_version_key_hash
#     ,bigint(substr(xxhash64(concat_ws('|', linkrole)), 1, 18))          AS linkrole_bigint_key
#     ,sha2(concat_ws('|', linkrole), 256)                                AS linkrole_key_hash
#     ,gaap_version
#     ,linkrole
#     ,leaf_node                                                           AS terse_label
#     ,level_1                                                             AS terse_label_level_1
#     ,level_2                                                             AS terse_label_level_2
#     ,level_3                                                             AS terse_label_level_3
#     ,level_4                                                             AS terse_label_level_4
#     ,level_5                                                             AS terse_label_level_5
#     ,level_6                                                             AS terse_label_level_6
#     ,level_7                                                             AS terse_label_level_7
#     ,level_8                                                             AS terse_label_level_8
#     ,level_9                                                             AS terse_label_level_9
#     ,level_10                                                            AS terse_label_level_10
#     ,level_11                                                            AS terse_label_level_11
# from df
# """)

# final_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{target_catalog}.{schema}.{table_name}")

In [0]:
from pyspark.sql import functions as F

df = spark.table("operations.finance_staging.dim_taxonomy_staging") \
    .select("linkrole", "gaap_version", "child_label", "parent_label") \
    .dropDuplicates()

# Initialize hierarchy
hierarchy_df = df.withColumn("level_1", F.col("parent_label")) \
    .select("linkrole", "gaap_version", "child_label", "level_1")

max_levels = 20

current_df = hierarchy_df

for i in range(2, max_levels + 1):
    parent_alias = f"p{i}"

    next_df = current_df.alias("c") \
        .join(
            df.alias(parent_alias),
            (
                (F.col(f"c.level_{i-1}") == F.col(f"{parent_alias}.child_label")) &
                (F.col("c.linkrole") == F.col(f"{parent_alias}.linkrole")) &
                (F.col("c.gaap_version") == F.col(f"{parent_alias}.gaap_version"))
            ),
            "left"
        ) \
        .withColumn(f"level_{i}", F.col(f"{parent_alias}.parent_label")) \
        .select("c.*", f"level_{i}")

    current_df = next_df

In [0]:
from pyspark.sql import functions as F

# Get all level columns
level_cols = [c for c in current_df.columns if c.startswith("level_")]

# Sort them numerically
level_cols_sorted = sorted(level_cols, key=lambda x: int(x.split("_")[1]))

# Reverse order (so highest level comes first)
reversed_levels = level_cols_sorted[::-1]

# Build new column expressions
new_cols = [
    F.col("linkrole"),
    F.col("gaap_version"),
    F.col("child_label")
]

for i, col_name in enumerate(reversed_levels, start=1):
    new_cols.append(F.col(col_name).alias(f"level_{i}"))

# Select reordered columns
final_df = current_df.select(*new_cols)

In [0]:
from pyspark.sql import functions as F

num_levels = len(reversed_levels)

# Create array of levels
final_df = final_df.withColumn(
    "levels_array",
    F.array(*[F.col(f"level_{i}") for i in range(1, num_levels + 1)])
)

# Remove nulls
final_df = final_df.withColumn(
    "levels_array",
    F.expr("filter(levels_array, x -> x is not null)")
)

# Re-expand safely using get()
for i in range(num_levels):
    final_df = final_df.withColumn(
        f"level_{i+1}",
        F.expr(f"get(levels_array, {i})")
    )

# Drop helper column
final_df = final_df.drop("levels_array")

In [0]:
final_df.createOrReplaceTempView('df')

In [0]:
final_df = spark.sql(f"""
select
     bigint(substr(xxhash64(concat_ws('|', child_label)), 1, 18))        AS terse_label_bigint_key
    ,sha2(concat_ws('|', child_label), 256)                               AS terse_label_key_hash
    ,bigint(substr(xxhash64(concat_ws('|', gaap_version)), 1, 18))      AS gaap_version_bigint_key
    ,sha2(concat_ws('|', gaap_version), 256)                            AS gaap_version_key_hash
    ,bigint(substr(xxhash64(concat_ws('|', linkrole)), 1, 18))          AS linkrole_bigint_key
    ,sha2(concat_ws('|', linkrole), 256)                                AS linkrole_key_hash
    ,gaap_version
    ,linkrole
    ,child_label                                                          AS terse_label
    ,level_1                                                             AS terse_label_level_1
    ,level_2                                                             AS terse_label_level_2
    ,level_3                                                             AS terse_label_level_3
    ,level_4                                                             AS terse_label_level_4
    ,level_5                                                             AS terse_label_level_5
    ,level_6                                                             AS terse_label_level_6
    ,level_7                                                             AS terse_label_level_7
    ,level_8                                                             AS terse_label_level_8
    ,level_9                                                             AS terse_label_level_9
    ,level_10                                                            AS terse_label_level_10
    ,level_11                                                            AS terse_label_level_11
    ,level_12                                                            AS terse_label_level_12
    ,level_13                                                            AS terse_label_level_13
    ,level_14                                                            AS terse_label_level_14
    ,level_15                                                            AS terse_label_level_15
    ,level_16                                                            AS terse_label_level_16
    ,level_17                                                            AS terse_label_level_17
    ,level_18                                                            AS terse_label_level_18
    ,level_19                                                            AS terse_label_level_19
    ,level_20                                                            AS terse_label_level_20
from df
""")

In [0]:
# final_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{target_catalog}.{schema}.{table_name}")